## Управление умной лампой жестами
## Smart lamp gesture control
 
### Цель: Реализовать механизм управления лампой через жесты и распознавание жестов
### Goal: Implement gesture recognition and control for a smart lightbulb

### Суть алгоритма:
Определение предметов на фоне.
  1) Берем снимок заднего фона.
  2) Сравниваем все последующие фреймы и находим разницу.
  3) Подбираем порог, чтобы определить необходимый предмет.
 
При этом фокусируемся только на определенной части изображения, не обращая внимания на другие предметы. Это увеличит скорость работы и точность алгоритма.
 
Также у нас должно быть несколько приоритетных жестов:
  * движение всей руки (тогда нам не нужно считать пальцы, высокий приоритет)
  * движение пальцев (низкий приоритет)
 
### Algorithm Core:
 
Background object detection.
1) Capture a background snapshot.
2) Compare all subsequent frames to the background to find the difference.
3) Apply a threshold to isolate the target object.
 
By focusing exclusively on a specific region of interest (ROI), we ignore irrelevant objects. This improves both processing speed and algorithm accuracy.
 
We also establish a hierarchy of priority gestures:
* Hand movement / Waving (high priority: does not require finger counting)
* Finger movement / Hand posture (low priority)
 
---

### Первым делом импортируем библиотеки.
 
### First things first, let's import libraries.

In [ ]:
import numpy as np
import cv2
import threading
from cv2 import CHAIN_APPROX_SIMPLE, RETR_EXTERNAL

Настроим необходимые переменные
 
Setting up variables

In [ ]:
bg = None
hand = None

frames_elapsed = 0
FRAME_HEIGHT = 500
FRAME_WIDTH = 600

CALIBRATION_TIME = 40
BG_WEIGHT = 0.5
DETECTION_THRESHOLD = 13

region_top = 0
region_bottom = int(2 * FRAME_HEIGHT / 3)
region_left  = int(FRAME_WIDTH / 2)
region_right = FRAME_WIDTH

LAMP_KEY= 0 # your lamp key
LAMP_IP = 0# your lamp ip
LAMP_ID = 0# your lamp id

lamp_is_on = True
lamp_lock = threading.Lock()

screen = cv2.VideoCapture(0)
screen.set(cv2.CAP_PROP_AUTOFOCUS, 0)

Подключаемся к лампе по протоколу tuya
 
Connecting to the lightbulb using the Tuya protocol

In [ ]:
import tinytuya

d = tinytuya.BulbDevice(LAMP_ID, LAMP_IP, LAMP_KEY, version=3.5)
print(d.status())

In [ ]:
class HandData:
    top = (0,0)
    bottom = (0,0)
    left = (0,0)
    right = (0,0)
    center = (0,0)
    prevCenter = (0,0)
    radius = 0
    isInFrame = False
    isWaving = False
    fingers = 0
    
    def __init__(self, top, bottom, left, right, center, radius):
        self.top = top
        self.bottom = bottom
        self.left = left
        self.right = right
        self.center = center
        self.radius = radius
        self.prevCenter = (0,0)
        self.isInFrame = False
        self.isWaving = False
        
    def update(self, top, bottom, left, right, radius):
        self.top = top
        self.bottom = bottom
        self.left = left
        self.right = right
        self.radius = radius
        
    def check_for_waving(self, center):
        self.prevCenter = self.center
        self.center = center
        
        distance = np.sqrt((self.center[0] - self.prevCenter[0])**2 + (self.center[1] - self.prevCenter[1])**2)
        
        self.isWaving = True if distance > 5 else False

### Функция для выключения лампы
### Function for toggling the lamp state

In [ ]:
def async_lamp_on(state: bool):
    global lamp_is_on
    global lamp_lock
    
    with lamp_lock:
        if state == False:
            d.turn_off()
        else: 
            d.turn_on()
        
        lamp_is_on = state

### Способ выводить статус на экран:
- "Calibrating..." для калибровки
- "Undefined" если рука не найдена
- "Waving" если пользователь машет
- Название жеста на английском

### Display status on the screen:
- "Calibrating..." for background calibration
- "Undefined" if no hand is detected
- "Waving" if the user waves their hand
- Name of the recognized gesture (Fist, Pointer, Peace)

In [ ]:
def write_on_image(frame, text):
    cv2.putText(frame, text, (10, 20), cv2.FONT_ITALIC, 0.5, (67, 0, 0))
    
    cv2.rectangle(frame, (region_left, region_top),
                (region_right, region_bottom), (0, 255, 0), 2)

In [ ]:
def update_hand_write_thresholded(region):
    region_pair = subtract(region)
    if region_pair is not None:
        (thresholded_region, segmented_region) = region_pair
        cv2.drawContours(region, [segmented_region], -1, (255, 255, 255))
        cv2.imshow('Segmented Image', region)
        
        get_hand(thresholded_region, segmented_region)

def update(frame):
    global hand
    global lamp_is_on
    
    region = get_region(frame)
    if frames_elapsed < CALIBRATION_TIME:
        get_average_bg(region)
    else:
        update_hand_write_thresholded(region)
    
    text = 'Searching...'
    
    if cv2.waitKey(1) == ord('o'):
        threading.Thread(target=d.turn_on, daemon=True).start()
        lamp_is_on = True
    
    if frames_elapsed < CALIBRATION_TIME:
        text = 'Calibrating...'
    elif hand == None or hand.isInFrame == False:
        text = "Undefined"
    else:
        if hand.isWaving:
            text = "Waving"
        elif hand.fingers == 0:
            text = "Fist"
        elif hand.fingers == 1:
            text = "Pointer"
        elif hand.fingers == 2:
            text = "Peace"
            
            if not lamp_lock.locked():
                t = threading.Thread(target=async_lamp_on, args=(False,),  daemon=True)
                t.start()
            
    write_on_image(frame, text)

---

### Во время калибровки мы сохраним цвет заднего фона, чтобы отделять руку от него.
 
Отделить руку проще всего по краям (edge detection). Там, где яркость изображения резко меняется, скорее всего, будет какой-то объект. Из-за этой яркости нам стоит перевести изображение в ч/б. Чтобы избежать "одиноких" пикселей, размываем изображение (gaussian blur).
 
Все вышеперечисленное выполнит следующая функция.

### Background calibration and edge isolation.
Converting the region of interest to grayscale, applying Gaussian blur and morphological closing to remove stray pixels and noise before background subtraction.

In [ ]:
def get_region(frame):
    region = frame[region_top:region_bottom,
                    region_left:region_right]
    region = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    region = cv2.GaussianBlur(region, (5,5), 0)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    region = cv2.morphologyEx(region, cv2.MORPH_CLOSE, kernel)
    return region

def get_average_bg(region):
    global bg
    if bg is None:
        bg = region.copy().astype('float')
        return
    cv2.accumulateWeighted(region, bg, BG_WEIGHT)

### Ключевая точка алгоритма - разница кадров
### Algorithm Core - Frame Differencing

In [ ]:
def subtract(region):
    global bg
    global hand
    diff = cv2.absdiff(bg.astype(np.uint8), region)
    
    thresholded_region = cv2.threshold(diff, DETECTION_THRESHOLD, 255, cv2.THRESH_BINARY)[1]
    
    (contours, _) = cv2.findContours(thresholded_region.copy(),
                                        cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
    
    if len(contours) == 0:
        if hand is not None:
            hand.isInFrame= False
        return
    
    else:
        if hand is not None:
            hand.isInFrame = True
        segmented_region = max(contours, key=cv2.contourArea)
        return (thresholded_region, segmented_region)

---
 
### Первый жест - "помахать"
Находить центр исходя из размеров объекта ненадёжно, поскольку большой палец может сильно сместить центр. Поэтому найдем точку внутри контура, максимально удалённую от крайних точек контура. Максимальное значение функции будет равно радиусу вписанной в ладонь окружности.

### First Gesture - "Waving"
Calculating the palm center using the Distance Transform to find the inner point furthest from the edges. The maximum value defines the radius of the palm's inscribed circle, ensuring tracking stability regardless of finger extension.

In [ ]:
def get_hand(thresholded_image, segmented_image):
    global hand
    convexHull = cv2.convexHull(segmented_image)
    
    top = tuple(convexHull[convexHull[:, :,1].argmin()][0])
    bottom = tuple(convexHull[convexHull[:, :,1].argmax()][0])
    left = tuple(convexHull[convexHull[:, :,0].argmin()][0])
    right = tuple(convexHull[convexHull[:, :,0].argmax()][0])
    
    distances = cv2.distanceTransform(thresholded_image, cv2.DIST_L2, 5)
    
    _, maxVal, _, maxLoc = cv2.minMaxLoc(distances)
    
    center = maxLoc
    radius = maxVal
    
    if hand == None:
        hand = HandData(top, bottom, left, right, center, radius)
    else:
        hand.update(top, bottom, left, right, radius)
        
    if frames_elapsed % 8 == 0:
        hand.check_for_waving(center)
        
    detected_fingers = count_fingers(segmented_image)
    
    if detected_fingers == 0:
        distance_to_top = np.linalg.norm(np.array(center) - np.array(top))
        if distance_to_top > 1.6 * radius:
            detected_fingers = 1
        else:
            detected_fingers = 0
    
    hand.fingers = detected_fingers

---
 
### Подсчёт пальцев
1. Сглаживаем контур для исключения самопересечений.
2. Ищем дефекты выпуклости для определения впадин между пальцами.
3. Рассчитываем углы впадин для фильтрации ложных срабатываний.
4. Кол-во пальцев = количество валидных впадин + 1.

### Counting fingers
1. Approximate the contour to eliminate self-intersections and noise.
2. Find convexity defects to locate spaces between fingers.
3. Calculate angles of defects to filter out non-finger spaces.
4. Finger count = defect spaces + 1.

In [ ]:
def count_fingers(segmented_image):
    fingers = 0
    
    epsilon = 0.01 * cv2.arcLength(segmented_image, True)
    approx_contour = cv2.approxPolyDP(segmented_image, epsilon, True)
    
    convexHull = cv2.convexHull(approx_contour, returnPoints=False)
    
    if convexHull is None or len(convexHull) < 3:
        return 0
        
    defects = cv2.convexityDefects(approx_contour, convexhull=convexHull)
    
    if defects is None:
        return 0
    
    for i in range(defects.shape[0]):
        s, e, f, d = defects[i, 0]
        start = tuple(approx_contour[s][0])
        end = tuple(approx_contour[e][0])
        far = tuple(approx_contour[f][0])
        
        a = np.linalg.norm(np.array(end) - np.array(start))
        b = np.linalg.norm(np.array(far) - np.array(start))
        c = np.linalg.norm(np.array(end) - np.array(far))
        
        if b * c == 0:
            continue
            
        cosine_value = (b*b + c*c - a*a) / (2 * b * c)
        cosine_value = np.clip(cosine_value, -1.0, 1.0)
        alpha = np.arccos(cosine_value)
        
        radius = hand.radius if hand is not None else 0
        
        if alpha <= np.pi / 2 and d > 0.3 * radius:
            fingers += 1
            
    if fingers > 0:
        fingers += 1
        
    return fingers

### Вот как выглядит главная функция, в которой происходит вся логика программы.
### Here is main function where all the things happen

In [ ]:
while True:
    _, frame = screen.read()
    frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
    
    frame = cv2.flip(frame, 1)
    
    update(frame)
    
    cv2.imshow("Webcam", frame)
    frames_elapsed += 1
    if cv2.waitKey(5) == ord('x'):
        break
    
screen.release()
cv2.destroyAllWindows()

### Итог: модель распознаёт жесты, однако она очень сильно зависит от освещения, что меня не устраивает. Реализую жесты через google mediapipe.
### Conclusion: Although the model successfully recognizes gestures, its heavy reliance on lighting conditions is a major drawback for me. Moving forward, I will implement gesture control using Google MediaPipe.